In [13]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [14]:
import json
from pathlib import Path

def load_json_or_jsonl(path):
    path = Path(path)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if not text:
        return []

    # First try normal JSON
    try:
        data = json.loads(text)
        if isinstance(data, dict) and "data" in data:
            return data["data"]
        if isinstance(data, list):
            return data
        return [data]
    except json.JSONDecodeError:
        pass

    # Fallback: JSON Lines
    records = []
    for line in text.splitlines():
        line = line.strip()
        if line:
            records.append(json.loads(line))
    return records


In [15]:
def flatten_recovery_records(records):
    rows = []

    for i, rec in enumerate(records):
        sequence_id = rec.get("sequence_id", f"seq_{i:06d}")
        source = rec.get("source", "")
        reference = rec.get("reference", "")

        for step in rec.get("recover_steps", []):
            rows.append({
                "sequence_id": sequence_id,
                "source": source,
                "reference": reference,
                "step_index": step.get("step_index"),
                "timestep": step.get("timestep"),
                "recover": step.get("recover", "")
            })

    return pd.DataFrame(rows)

In [16]:
records = load_json_or_jsonl("generation_outputs/diffuseq_qqp_h128_lr0.0001_t2000_sqrt_lossaware_seed102_qqp20260422-17:21:27/ema_0.9999_030000.pt.samples/seed123_step0.intermediate.json")
df = flatten_recovery_records(records)
df

,sequence_id,source,reference,step_index,timestep,recover
0,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i'm a triple capricorn ( sun, moon and a...",0,1999,[CLS] astro legitimate through legitimate scra...
1,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i'm a triple capricorn ( sun, moon and a...",1,1998,[CLS] welcomed safely i legitimate elected ele...
2,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i'm a triple capricorn ( sun, moon and a...",2,1997,[CLS] how safely i elected bother elected plea...
3,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i'm a triple capricorn ( sun, moon and a...",3,1996,[CLS] welcomed safely i legitimate please elec...
4,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i'm a triple capricorn ( sun, moon and a...",4,1995,[CLS] welcomed safely i legitimate elected ele...
...,...,...,...,...,...,...
39995,seq_000019,[CLS] how is the new harry potter book'harry p...,[CLS] how bad is the new book by j. k rowling?...,1995,4,[CLS] did the new behind book harry'potter har...
39996,seq_000019,[CLS] how is the new harry potter book'harry p...,[CLS] how bad is the new book by j. k rowling?...,1996,3,[CLS] did the new behind book harry'potter har...
39997,seq_000019,[CLS] how is the new harry potter book'harry p...,[CLS] how bad is the new book by j. k rowling?...,1997,2,[CLS] did the new behind book harry'potter har...
39998,seq_000019,[CLS] how is the new harry potter book'harry p...,[CLS] how bad is the new book by j. k rowling?...,1998,1,[CLS] did the new behind book harry'potter har...


In [17]:
SPECIAL_TOKENS = [
    "[CLS]",
    "[SEP]",
    "[PAD]",
    "[UNK]",
    "[MASK]"
]

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    for tok in SPECIAL_TOKENS:
        text = text.replace(tok, " ")

    text = re.sub(r"\s+", " ", text).strip()
    return text

In [18]:
df["source_clean"] = df["source"].apply(clean_text)
df["reference_clean"] = df["reference"].apply(clean_text)
df["recover_clean"] = df["recover"].apply(clean_text)

df[["sequence_id", "step_index", "timestep", "recover_clean"]].head()

,sequence_id,step_index,timestep,recover_clean
0,seq_000000,0,1999,astro legitimate through legitimate scrap plea...
1,seq_000000,1,1998,welcomed safely i legitimate elected elected p...
2,seq_000000,2,1997,how safely i elected bother elected please ple...
3,seq_000000,3,1996,welcomed safely i legitimate please elected pl...
4,seq_000000,4,1995,welcomed safely i legitimate elected elected p...


In [19]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_unique_texts(df, text_columns, batch_size=128):
    unique_texts = pd.unique(
        pd.concat([df[col] for col in text_columns], ignore_index=True)
        .dropna()
        .astype(str)
    )

    unique_texts = [t for t in unique_texts if t.strip()]

    embeddings = model.encode(
        unique_texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    return dict(zip(unique_texts, embeddings))

In [20]:
embedding_lookup = embed_unique_texts(
    df,
    text_columns=["source_clean", "reference_clean", "recover_clean"],
    batch_size=128
)

def cosine_from_lookup(df, left_col, right_col, embedding_lookup):
    scores = np.full(len(df), np.nan)

    mask = (
        df[left_col].astype(str).str.strip().ne("") &
        df[right_col].astype(str).str.strip().ne("")
    )

    left_embeddings = np.vstack(
        df.loc[mask, left_col].map(embedding_lookup).to_numpy()
    )

    right_embeddings = np.vstack(
        df.loc[mask, right_col].map(embedding_lookup).to_numpy()
    )

    scores[mask.to_numpy()] = np.sum(left_embeddings * right_embeddings, axis=1)

    return scores

df["sim_recover_reference"] = cosine_from_lookup(
    df,
    "recover_clean",
    "reference_clean",
    embedding_lookup
)

df["sim_recover_source"] = cosine_from_lookup(
    df,
    "recover_clean",
    "source_clean",
    embedding_lookup
)

df["sim_source_reference"] = cosine_from_lookup(
    df,
    "source_clean",
    "reference_clean",
    embedding_lookup
)

df["sim_recover_reference_pct"] = (df["sim_recover_reference"] * 100).round(2)
df["sim_recover_source_pct"] = (df["sim_recover_source"] * 100).round(2)
df["sim_source_reference_pct"] = (df["sim_source_reference"] * 100).round(2)

df["avg_reference_source_similarity_pct"] = (
    df[["sim_recover_reference_pct", "sim_recover_source_pct"]]
    .mean(axis=1)
    .round(2)
)

df["gain_over_source_reference"] = (
    df["sim_recover_reference"] - df["sim_source_reference"]
).round(4)

df["gain_over_source_reference_pct"] = (
    df["gain_over_source_reference"] * 100
).round(2)

best_by_sequence = (
    df.sort_values(
        ["sequence_id", "sim_recover_reference"],
        ascending=[True, False]
    )
    .groupby("sequence_id", as_index=False)
    .first()
)

best_by_sequence[
    [
        "sequence_id",
        "step_index",
        "timestep",
        "sim_recover_reference_pct",
        "sim_recover_source_pct",
        "recover_clean"
    ]
].head()

df.to_parquet("scored_recovery_steps.parquet_ecc", index=False)
best_by_sequence.to_parquet("best_recovery_by_sequence_ecc.parquet", index=False)

Batches: 100%|██████████| 14/14 [00:00<00:00, 94.72it/s]


In [21]:

path = "/export/usuarios01/zhuldyz/Zhuldyz_files/DiffuSeq/best_recovery_by_sequence_ecc.parquet"

df = pd.read_parquet(path)
df.head()

,sequence_id,source,reference,step_index,timestep,recover,source_clean,reference_clean,recover_clean,sim_recover_reference,sim_recover_source,sim_source_reference,sim_recover_reference_pct,sim_recover_source_pct,sim_source_reference_pct,avg_reference_source_similarity_pct,gain_over_source_reference,gain_over_source_reference_pct
0,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i'm a triple capricorn ( sun, moon and a...",179,1820,[CLS] how can i am aric moon cap sun cap bothe...,astrology : i am a capricorn sun cap moon and ...,"i'm a triple capricorn ( sun, moon and ascenda...",how can i am aric moon cap sun cap bother that...,0.550814,0.676002,0.837468,55.08,67.60,83.75,61.34,-0.2867,-28.67
1,seq_000001,[CLS] how can i be a good geologist? [SEP] [SEP],[CLS] what should i do to be a great geologist...,10,1989,[CLS] how can i be a good good geologist? [SEP],how can i be a good geologist?,what should i do to be a great geologist?,how can i be a good good geologist?,0.926248,0.992908,0.936959,92.62,99.29,93.70,95.96,-0.0107,-1.07
2,seq_000002,[CLS] how do i read and find my youtube commen...,[CLS] how can i see all my youtube comments? [...,74,1925,[CLS] how do i read read my youtube comments?,how do i read and find my youtube comments?,how can i see all my youtube comments?,how do i read read my youtube comments?,0.783558,0.917174,0.879711,78.36,91.72,87.97,85.04,-0.0962,-9.62
3,seq_000003,[CLS] what can make physics easy to learn? [SE...,[CLS] how can you make physics easy to learn? ...,280,1719,[CLS] what are some legitimate easy to physics...,what can make physics easy to learn?,how can you make physics easy to learn?,what are some legitimate easy to physics make ...,0.843516,0.865177,0.974253,84.35,86.52,97.43,85.44,-0.1307,-13.07
4,seq_000004,[CLS] what was your first sexual experience li...,[CLS] what was your first sexual experience? [...,163,1836,[CLS] what was your sexual you'and have ( s se...,what was your first sexual experience like?,what was your first sexual experience?,what was your sexual you'and have ( s sexual's...,0.821137,0.817322,0.971795,82.11,81.73,97.18,81.92,-0.1507,-15.07


In [22]:
df

,sequence_id,source,reference,step_index,timestep,recover,source_clean,reference_clean,recover_clean,sim_recover_reference,sim_recover_source,sim_source_reference,sim_recover_reference_pct,sim_recover_source_pct,sim_source_reference_pct,avg_reference_source_similarity_pct,gain_over_source_reference,gain_over_source_reference_pct
0,seq_000000,[CLS] astrology : i am a capricorn sun cap moo...,"[CLS] i'm a triple capricorn ( sun, moon and a...",179,1820,[CLS] how can i am aric moon cap sun cap bothe...,astrology : i am a capricorn sun cap moon and ...,"i'm a triple capricorn ( sun, moon and ascenda...",how can i am aric moon cap sun cap bother that...,0.550814,0.676002,0.837468,55.08,67.60,83.75,61.34,-0.2867,-28.67
1,seq_000001,[CLS] how can i be a good geologist? [SEP] [SEP],[CLS] what should i do to be a great geologist...,10,1989,[CLS] how can i be a good good geologist? [SEP],how can i be a good geologist?,what should i do to be a great geologist?,how can i be a good good geologist?,0.926248,0.992908,0.936959,92.62,99.29,93.70,95.96,-0.0107,-1.07
2,seq_000002,[CLS] how do i read and find my youtube commen...,[CLS] how can i see all my youtube comments? [...,74,1925,[CLS] how do i read read my youtube comments?,how do i read and find my youtube comments?,how can i see all my youtube comments?,how do i read read my youtube comments?,0.783558,0.917174,0.879711,78.36,91.72,87.97,85.04,-0.0962,-9.62
3,seq_000003,[CLS] what can make physics easy to learn? [SE...,[CLS] how can you make physics easy to learn? ...,280,1719,[CLS] what are some legitimate easy to physics...,what can make physics easy to learn?,how can you make physics easy to learn?,what are some legitimate easy to physics make ...,0.843516,0.865177,0.974253,84.35,86.52,97.43,85.44,-0.1307,-13.07
4,seq_000004,[CLS] what was your first sexual experience li...,[CLS] what was your first sexual experience? [...,163,1836,[CLS] what was your sexual you'and have ( s se...,what was your first sexual experience like?,what was your first sexual experience?,what was your sexual you'and have ( s sexual's...,0.821137,0.817322,0.971795,82.11,81.73,97.18,81.92,-0.1507,-15.07
5,seq_000005,[CLS] what would a trump presidency mean for c...,[CLS] how will a trump presidency affect the s...,829,1170,[CLS] what does a trump for presidency interna...,what would a trump presidency mean for current...,how will a trump presidency affect the student...,what does a trump for presidency international...,0.700181,0.684159,0.662187,70.02,68.42,66.22,69.22,0.0380,3.80
6,seq_000006,[CLS] what does manipulation mean? [SEP] [SEP],[CLS] what does manipulation means? [SEP],0,1999,[CLS] what does manipulation ” mean? [SEP],what does manipulation mean?,what does manipulation means?,what does manipulation ” mean?,0.980142,0.988496,0.989422,98.01,98.85,98.94,98.43,-0.0093,-0.93
7,seq_000007,[CLS] why are so many quora users posting ques...,[CLS] why do people ask quora questions which ...,795,1204,[CLS] why did many here questions usersra up t...,why are so many quora users posting questions ...,why do people ask quora questions which can be...,why did many here questions usersra up that fo...,0.702740,0.691990,0.915100,70.27,69.20,91.51,69.74,-0.2124,-21.24
8,seq_000008,[CLS] why do rockets look white? [SEP] [SEP],[CLS] why are rockets and boosters painted whi...,691,1308,[CLS] why do rockets want look white white? [SEP],why do rockets look white?,why are rockets and boosters painted white?,why do rockets want look white white?,0.861292,0.958263,0.859587,86.13,95.83,85.96,90.98,0.0017,0.17
9,seq_000009,[CLS] how should i prepare for ca final law? [...,[CLS] how one should know that he / she comple...,37,1962,[CLS] how should i prepare ca for final law? [...,how should i prepare for ca final law?,how one should know that he / she completely p...,how should i prepare ca for final law?,0.689763,0.981453,0.675563,68.98,98.15,67.56,83.56,0.0142,1.42
